# Visualize 25k Sample — Train/Test Split

Produces 2 GeoTIFF maps (only sampled parcels, no background):
1. **full_sample.tif** — All 25k sampled parcels: not-vacant vs. vacant
2. **train_test_split.tif** — Train/test split with vacant highlighted in each

In [1]:
from pathlib import Path

from sklearn.model_selection import train_test_split

from vacant_lot.analysis import read_csv_from_gcs
from vacant_lot.config import load_config
from vacant_lot.data_utils import load_gdb
from vacant_lot.raster_utils import rasterize_categorical_map

REPO_ROOT = Path.cwd().parent
FIGURES_DIR = REPO_ROOT / "modeling/outputs/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
cfg = load_config("nyc_buildings.yaml", config_dir=REPO_ROOT / "config")

18:49:23 | INFO | vacant_lots | Loaded config for: nyc_buildings.yaml


## Load data

In [3]:
# Load all NYC parcels (~857k), keep only the 25k in our sample
full_gdf = load_gdb(cfg.get_parcel_path(REPO_ROOT), layer=cfg.parcel.layer)
full_gdf = full_gdf[["BBL", cfg.parcel.landuse_column, "geometry"]].to_crs(cfg.raster.output_crs)
print(f"Loaded {len(full_gdf):,} total parcels")

Loaded 856,998 total parcels


In [4]:
# Load spectral stats to get sampled BBLs
spectral_df = read_csv_from_gcs(
    cfg.gcp.bucket,
    f"{cfg.gee.export_prefix}/parcel_spectral_stats.csv",
)
sampled_bbls = set(spectral_df["BBL"])
print(f"Sampled BBLs: {len(sampled_bbls):,}")

18:49:30 | INFO | vacant_lots | Reading CSV from gs://thesis_parcels/eda/new_york_new_york_bldgclss/parcel_spectral_stats.csv
18:49:32 | INFO | vacant_lots | Loaded 25000 rows, 35 columns
Sampled BBLs: 25,000


In [5]:
# Filter to only sampled parcels
in_sample = full_gdf["BBL"].isin(sampled_bbls)
sample_gdf = full_gdf[in_sample].copy()
is_vacant = sample_gdf[cfg.parcel.landuse_column].isin(cfg.parcel.vacant_codes)

print(f"Sampled parcels: {len(sample_gdf):,}")
print(f"Vacant in sample: {is_vacant.sum():,}")

Sampled parcels: 25,000
Vacant in sample: 500


In [6]:
# Train/test split (80/20, stratified by vacant label)
sample_gdf["is_vacant"] = is_vacant.astype(int)

train_bbls_series, test_bbls_series = train_test_split(
    sample_gdf["BBL"],
    test_size=0.2,
    stratify=sample_gdf["is_vacant"],
    random_state=42,
)
train_bbls = set(train_bbls_series)
test_bbls = set(test_bbls_series)

print(f"Train: {len(train_bbls):,}, Test: {len(test_bbls):,}")

Train: 20,000, Test: 5,000


## TIF 1: Full 25k sample (not-vacant vs. vacant)

In [7]:
sample_gdf["cat"] = 1  # not vacant
sample_gdf.loc[is_vacant, "cat"] = 2  # vacant

rasterize_categorical_map(
    sample_gdf, "cat",
    palette={1: (196, 139, 159), 2: (255, 224, 38)},
    output_path=FIGURES_DIR / "full_sample.tif",
)

18:49:32 | INFO | vacant_lots | Rasterizing categorical map to /path/to/Vacant_Lot_Detection/modeling/outputs/figures/full_sample.tif
18:49:32 | INFO | vacant_lots | Raster dimensions: 15501 x 15505 px, resolution: 3.0
18:49:39 | INFO | vacant_lots | Written: /path/to/Vacant_Lot_Detection/modeling/outputs/figures/full_sample.tif (15501x15505 px, 11.0 MB)


PosixPath('/path/to/Vacant_Lot_Detection/modeling/outputs/figures/full_sample.tif')

## TIF 2: Train/test split

- Train, not vacant → mauve
- Train, vacant → yellow
- Test, not vacant → blue
- Test, vacant → cyan

In [8]:
in_train = sample_gdf["BBL"].isin(train_bbls)
in_test = sample_gdf["BBL"].isin(test_bbls)

sample_gdf["cat"] = 0
sample_gdf.loc[in_train & ~is_vacant, "cat"] = 1  # train, not vacant
sample_gdf.loc[in_train & is_vacant, "cat"] = 2   # train, vacant
sample_gdf.loc[in_test & ~is_vacant, "cat"] = 3   # test, not vacant
sample_gdf.loc[in_test & is_vacant, "cat"] = 4    # test, vacant

rasterize_categorical_map(
    sample_gdf[sample_gdf["cat"] > 0], "cat",
    palette={
        1: (196, 139, 159),  # train, not vacant — mauve
        2: (255, 224, 38),   # train, vacant — yellow
        3: (100, 140, 200),  # test, not vacant — blue
        4: (0, 230, 210),    # test, vacant — cyan
    },
    output_path=FIGURES_DIR / "train_test_split.tif",
)

18:49:39 | INFO | vacant_lots | Rasterizing categorical map to /path/to/Vacant_Lot_Detection/modeling/outputs/figures/train_test_split.tif
18:49:39 | INFO | vacant_lots | Raster dimensions: 15501 x 15505 px, resolution: 3.0
18:49:44 | INFO | vacant_lots | Written: /path/to/Vacant_Lot_Detection/modeling/outputs/figures/train_test_split.tif (15501x15505 px, 11.2 MB)


PosixPath('/path/to/Vacant_Lot_Detection/modeling/outputs/figures/train_test_split.tif')